# FinRAG QLoRA Fine-Tuning

This notebook fine-tunes `Qwen/Qwen2.5-7B-Instruct` with LoRA adapters on FinQA-style financial QA examples. Use a CUDA GPU runtime.

In [ ]:
import torch\nassert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > GPU before running.'\nprint(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')

Set `PROJECT_DIR` to this repository. If you opened the repo directly in a Colab-backed VS Code environment, use that mounted path instead.

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/Financial_QA_ChatBot'\n%cd $PROJECT_DIR

In [ ]:
!bash scripts/colab_setup.sh\n# If this still reports mixed CUDA versions, use Runtime > Disconnect and delete runtime, then reconnect.

In [ ]:
!PYTHONPATH=src python -m finrag.fine_tuning

In [ ]:
!PYTHONPATH=src python -m finrag.train_qlora \
  --model-name Qwen/Qwen2.5-7B-Instruct \
  --train-file data/fine_tuning/financial_qa_mix_train.jsonl \
  --output-dir /content/drive/MyDrive/finrag-adapters/qwen2_5_7b_financial_qa_lora \
  --epochs 1 \
  --max-length 1536 \
  --batch-size 1 \
  --gradient-accumulation-steps 16 \
  --learning-rate 2e-4 \
  --lora-r 16 \
  --lora-alpha 32

In [ ]:
!PYTHONPATH=src python -m finrag.hf_adapter_answer \
  --adapter-path /content/drive/MyDrive/finrag-adapters/qwen2_5_7b_financial_qa_lora \
  --model-name Qwen/Qwen2.5-7B-Instruct \
  'What risks did Apple report related to supply chains?'

## Serve Qwen From The Colab GPU Runtime

Local Streamlit should run on your Mac CPU. Colab should only host the Qwen 2.5 7B generation endpoint. Copy the public tunnel URL from the last cell and paste it into the local Streamlit app.

In [ ]:
!bash scripts/colab_start_qwen_server.sh

In [ ]:
# Force base model only, ignoring any saved adapter.
# !pkill -f 'finrag.qwen_server' || true
# !nohup env PYTHONPATH=src python -m finrag.qwen_server \
#   --model-name Qwen/Qwen2.5-7B-Instruct \
#   --port 8000 > qwen_server.log 2>&1 &

In [ ]:
%%bash\n# Expose the Colab Qwen server publicly so local Streamlit can call it.\npkill -f cloudflared || true\nwget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared\nchmod +x cloudflared\nnohup ./cloudflared tunnel --url http://127.0.0.1:8000 > cloudflared.log 2>&1 &\nsleep 8\ngrep -o 'https://[-a-zA-Z0-9.]*trycloudflare.com' cloudflared.log | tail -1

### Fallback: expose the Qwen server with ngrok\n\nUse this if Cloudflare Quick Tunnel returns a 500 error. You need a free ngrok auth token from https://dashboard.ngrok.com/get-started/your-authtoken.

In [ ]:
from pyngrok import ngrok\n\nNGROK_AUTHTOKEN = ''  # paste your ngrok token here\nif not NGROK_AUTHTOKEN:\n    raise ValueError('Paste your ngrok auth token into NGROK_AUTHTOKEN')\n\nngrok.set_auth_token(NGROK_AUTHTOKEN)\npublic_url = ngrok.connect(8000, 'http').public_url\nprint(public_url)